In [0]:
# List files in the directory
dbutils.fs.ls("/Workspace/Repos/Databricks_dev/sai_de/SampleData/")

# Or check using Python
# import os
# os.listdir("/dbfs/Workspace/Repos/Databricks_dev/sai_de/SampleData/")

In [0]:
df = spark.read.csv('../SampleData/healthcare_dataset.csv', header=True, inferSchema=True)

display(df)

## Job, Stages and Task

- **Job**: A high-level action (like reading or writing data) triggered by a Spark API call.
- **Stage**: A set of tasks that can be executed together, divided based on data shuffling.
- **Task**: The smallest unit of execution, representing a computation on a partition of data.

In [0]:
# Since DBFS access is not enabled in the free account, I uploaded the data to the testdb catalog and am reading it from there.

df1 = spark.read.table('testdb.testschema.healthcare_dataset')

display(df1)

In [0]:
# Turn off AQE
spark.conf.set("spark.sql.adaptive.enabled", "false")

# Verify it's disabled
print(spark.conf.get("spark.sql.adaptive.enabled"))

# ## Option 2: Disable at Cluster Level (Permanent)

# 1. Go to **Compute** in Databricks
# 2. Select your cluster
# 3. Click **Edit**
# 4. Scroll to **Advanced Options** → **Spark** tab
# 5. Add this to **Spark Config**:

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

df3 = df1.filter(df1['Medical Condition'] == 'Cancer')

In [0]:
df3 = df3.select('Name', 'Age', 'Blood Type', 'Medical Condition')

In [0]:
df3 = df3.groupBy('Blood Type').count()

# Spark Adaptive Query Execution (AQE)

AQE automatically optimizes Spark query plans at runtime using actual data statistics.

**Key benefits:**
- Adjusts join strategies dynamically (e.g., uses broadcast join for small tables).
- Merges small shuffle partitions to minimize task overhead.
- Handles skewed partitions to balance workload across tasks.

AQE is enabled by default in Databricks (`spark.sql.adaptive.enabled = true`).

**To disable AQE:**
python
spark.conf.set("spark.sql.adaptive.enabled", "false")


**To check AQE status:**
python
print(spark.conf.get("spark.sql.adaptive.enabled"))

# Example: Narrow vs Wide Transformations in Spark

- **Narrow transformation**: Operations like `filter` and `select` do not require data shuffling. Each executor processes its partition independently within a single stage.
  
  python
  df_narrow = df1.filter(df1['Medical Condition'] == 'Cancer') \
                 .select('Name', 'Age', 'Blood Type', 'Medical Condition')
  

- **Wide transformation**: Operations like `groupBy` trigger a shuffle, requiring data to be redistributed across partitions. This creates a new stage with tasks for each output partition.
  
  python
  df_wide = df_narrow.groupBy('Blood Type').count()
  

In [0]:
display(df3)